# Dimensionality-Preserving SSL - Training Launcher

This notebook is a thin launcher for the `dim-ssl` codebase.  
All code lives in the GitHub repo — this notebook just sets up the environment and kicks off training.

**Setup:**
1. Make sure GPU runtime is enabled: Runtime → Change runtime type → GPU
2. Replace `YOUR_GITHUB_USERNAME` with your actual GitHub username
3. Run all cells

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Clone repo (replace with your GitHub URL)
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/dim-ssl.git"

import os
if not os.path.exists("dim-ssl"):
    !git clone {REPO_URL}
else:
    !cd dim-ssl && git pull

%cd dim-ssl

In [ ]:
# Install dependencies
!pip install -e . -q

In [ ]:
# (Optional) Login to W&B for experiment tracking
# Get your API key from https://wandb.ai/authorize
import wandb
wandb.login()

## Train Baselines
Run these first to establish baseline numbers.

In [ ]:
# SimCLR baseline
!python scripts/train.py --config configs/simclr_cifar100.yaml

In [ ]:
# VICReg baseline
!python scripts/train.py --config configs/vicreg_cifar100.yaml

## Train with Dimensionality Regularizer
The main experimental runs.

In [ ]:
# SimCLR + dim reg
!python scripts/train.py --config configs/simclr_cifar100_dimreg.yaml

In [ ]:
# VICReg + dim reg
!python scripts/train.py --config configs/vicreg_cifar100_dimreg.yaml

## Quick Hyperparameter Sweep
Override dim_reg_weight via command line to find the sweet spot.

In [ ]:
# Sweep λ values for SimCLR
for lam in [0.01, 0.05, 0.1, 0.5, 1.0]:
    print(f"\n{'='*60}")
    print(f"Running SimCLR + dim_reg with lambda={lam}")
    print(f"{'='*60}\n")
    !python scripts/train.py \
        --config configs/simclr_cifar100_dimreg.yaml \
        --overrides dim_reg_weight={lam} seed=42

## Save Checkpoints to Google Drive
Important for Colab — sessions can die at any time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
drive_path = "/content/drive/MyDrive/dim-ssl-checkpoints"
os.makedirs(drive_path, exist_ok=True)
shutil.copytree("./checkpoints", drive_path, dirs_exist_ok=True)
print(f"Checkpoints saved to {drive_path}")